In [1]:
#  Imports and Setup

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import time
import json
import os
import sys
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from pathlib import Path
import os, sys
# Add the project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Change working directory to project root so config files can be found
os.chdir(project_root)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 16

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CPUs available: {os.cpu_count()}")

Device: cuda
GPU: Tesla V100-PCIE-16GB
GPU Memory: 16.9 GB
CPUs available: 96


In [ ]:
#  Benchmark Configuration

@dataclass
class BenchmarkConfig:
    """Configuration for speedup benchmarking."""
    # Batch sizes to test
    BATCH_SIZES: List[int] = field(default_factory=lambda: [1, 4, 16, 64, 256, 1024, 2048, 4096])
    
    # Number of repetitions for timing stability
    N_WARMUP: int = 5
    N_REPEATS: int = 20
    
    # FMU simulation parameters
    FMU_SYSTEM: str = "summit"
    FMU_STEP_SIZE: int = 1  # 1 second per step
    
    # Surrogate model parameters
    HISTORY_STEPS_P1: int = 40      # Phase 1 LSTM
    HISTORY_STEPS_P2: int = 40      # Phase 2 DeepONet
    HISTORY_STEPS_P3: int = 40      # Phase 3 DeepONet+Fixes
    HISTORY_STEPS_P4: int = 40      # Phase 4 Domain-Specific (min)
    HISTORY_STEPS_P5: int = 40      # Phase 5 Federated
    
    SUBSAMPLE_FACTOR: int = 30      # 30s per step
    
    # Number of CDUs
    NUM_CDUS: int = 257
    
    # Paths to saved models
    P1_MODEL_PATH: str = "./surrogate_models/saved_models/phase1_baseline_lstm/phase1_lstm_best.pth"
    P2_MODEL_PATH: str = "./surrogate_models/saved_models/phase2_basic_deeponet/phase2_deeponet_best.pth"
    P3_MODEL_PATH: str = "./surrogate_models/saved_models/phase3_hybrid_deeponet_fixes/phase3_deeponet_fixes_best.pth"
    P4_MODEL_DIR: str = "./surrogate_models/saved_models/phase4_domain_specific_deeponet"
    P5_MODEL_PATH: str = "./surrogate_models/saved_models/phase5_federated/phase5_model.pt"
    
    # Data path (for FMU input generation)
    DATA_PATH: str = "../data/summit/systematic/systematic_fmu_output_11hrs.parquet"


bench_config = BenchmarkConfig()
print("Benchmark configuration:")
print(f"  Batch sizes: {bench_config.BATCH_SIZES}")
print(f"  Warmup runs: {bench_config.N_WARMUP}")
print(f"  Timing runs: {bench_config.N_REPEATS}")
print(f"  NUM CDUs: {bench_config.NUM_CDUS}")

Benchmark configuration:
  Batch sizes: [1, 4, 16, 64, 256, 1024, 2048, 4096]
  Warmup runs: 5
  Timing runs: 20
  NUM CDUs: 257


In [3]:
#  Model Architecture Definitions (minimal, for loading weights)
# We redefine the model classes needed to load checkpoints.

# ─── Phase 1: Baseline LSTM ───────────────────────────────────────────────────

class BaselineLSTM(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=128,
                 num_layers=2, dropout=0.2, prediction_steps=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.prediction_steps = prediction_steps

        self.input_proj = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.lstm_norm = nn.LayerNorm(hidden_size)
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.Tanh(),
            nn.Linear(hidden_size // 2, 1),
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.LayerNorm(hidden_size * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size * prediction_steps),
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        lstm_out = self.lstm_norm(lstm_out)
        attn_w = torch.softmax(self.attention(lstm_out), dim=1)
        ctx = torch.sum(lstm_out * attn_w, dim=1)
        out = self.decoder(ctx)
        return out.view(B, self.prediction_steps, self.output_size)


# ─── Phase 2/3: Temporal DeepONet + Algebraic MLP ─────────────────────────────

class AlgebraicPathway(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=64, n_layers=3, dropout=0.1):
        super().__init__()
        layers = [nn.Linear(input_size, hidden_size), nn.LayerNorm(hidden_size),
                  nn.GELU(), nn.Dropout(dropout)]
        for _ in range(n_layers - 2):
            layers += [nn.Linear(hidden_size, hidden_size), nn.LayerNorm(hidden_size),
                       nn.GELU(), nn.Dropout(dropout)]
        layers.append(nn.Linear(hidden_size, output_size))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        B, K, _ = x.shape
        return self.mlp(x.reshape(B * K, -1)).reshape(B, K, -1)


class TemporalDeepONet(nn.Module):
    def __init__(self, input_size, output_size, prediction_steps,
                 lstm_hidden=64, trunk_hidden=64, n_basis=32, dropout=0.3):
        super().__init__()
        self.output_size = output_size
        self.prediction_steps = prediction_steps
        self.n_basis = n_basis

        self.branch_encoder = nn.LSTM(input_size=input_size, hidden_size=lstm_hidden,
                                       num_layers=2, batch_first=True, dropout=dropout)
        self.branch_norm = nn.LayerNorm(lstm_hidden)
        self.attention = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden // 2), nn.Tanh(),
            nn.Linear(lstm_hidden // 2, 1))
        self.branch_head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden), nn.LayerNorm(lstm_hidden),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(lstm_hidden, n_basis * output_size))

        self.register_buffer('query_times', torch.linspace(0, 1, prediction_steps).view(-1, 1))
        n_freqs = 8
        self.register_buffer('freqs', torch.linspace(1, n_freqs, n_freqs) * np.pi)
        trunk_in = 1 + 2 * n_freqs
        self.trunk_net = nn.Sequential(
            nn.Linear(trunk_in, trunk_hidden), nn.GELU(),
            nn.Linear(trunk_hidden, trunk_hidden), nn.GELU(),
            nn.Linear(trunk_hidden, n_basis))
        self.output_scale = nn.Parameter(torch.ones(output_size) * 0.1)
        self.output_bias = nn.Parameter(torch.zeros(output_size))

    def forward(self, x):
        B = x.shape[0]
        lstm_out, _ = self.branch_encoder(x)
        lstm_out = self.branch_norm(lstm_out)
        attn_w = torch.softmax(self.attention(lstm_out), dim=1)
        ctx = torch.sum(lstm_out * attn_w, dim=1)
        branch_out = self.branch_head(ctx).view(B, self.n_basis, self.output_size)
        t = self.query_times
        feats = torch.cat([t, torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        trunk_out = self.trunk_net(feats)
        out = torch.einsum('pn,bno->bpo', trunk_out, branch_out)
        return out * self.output_scale + self.output_bias


class HybridDeepONet_P2(nn.Module):
    def __init__(self, temporal_input_size, temporal_output_size,
                 algebraic_input_size, algebraic_output_size, prediction_steps, config_dict):
        super().__init__()
        self.temporal_pathway = TemporalDeepONet(
            temporal_input_size, temporal_output_size, prediction_steps,
            lstm_hidden=config_dict.get('DEEPONET_HIDDEN', 64),
            n_basis=config_dict.get('DEEPONET_N_BASIS', 32))
        self.algebraic_pathway = AlgebraicPathway(
            algebraic_input_size, algebraic_output_size,
            hidden_size=config_dict.get('ALGEBRAIC_HIDDEN', 64))

    def forward(self, x_temporal, x_algebraic):
        return self.temporal_pathway(x_temporal), self.algebraic_pathway(x_algebraic)


# ─── Phase 5: Federated DeepM&Mnet (simplified loader) ────────────────────────

class BranchNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers, n_basis, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.norm = nn.LayerNorm(hidden_size)
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.Tanh(),
            nn.Linear(hidden_size // 2, 1))
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.LayerNorm(hidden_size),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_size, n_basis))

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.norm(out)
        w = torch.softmax(self.attention(out), dim=1)
        ctx = torch.sum(out * w, dim=1)
        return self.projection(ctx)


class TBranchNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers, n_basis, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.norm = nn.LayerNorm(hidden_size)
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.Tanh(),
            nn.Linear(hidden_size // 2, 1))
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.LayerNorm(hidden_size),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_size, n_basis))

    def forward(self, y):
        out, _ = self.lstm(y)
        out = self.norm(out)
        w = torch.softmax(self.attention(out), dim=1)
        ctx = torch.sum(out * w, dim=1)
        return self.projection(ctx)


class FourierTrunkNetwork(nn.Module):
    def __init__(self, n_fourier, hidden_size, n_basis, prediction_steps):
        super().__init__()
        trunk_in = 1 + 2 * n_fourier
        self.net = nn.Sequential(
            nn.Linear(trunk_in, hidden_size), nn.GELU(),
            nn.Linear(hidden_size, hidden_size), nn.GELU(),
            nn.Linear(hidden_size, n_basis))
        self.register_buffer('freqs', torch.linspace(1, n_fourier, n_fourier) * np.pi)
        self.register_buffer('query_times', torch.linspace(0, 1, prediction_steps).view(-1, 1))

    def forward(self, K=None):
        t = self.query_times[:K] if K else self.query_times
        feats = torch.cat([t, torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        return self.net(feats)


class DecoderHead(nn.Module):
    def __init__(self, n_basis, hidden_dim, n_outputs, dropout, name=''):
        super().__init__()
        self.name = name
        self.n_outputs = n_outputs
        self.net = nn.Sequential(
            nn.Linear(n_basis, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_outputs))
        self.output_scale = nn.Parameter(torch.ones(n_outputs) * 0.1)

    def forward(self, z):
        return self.net(z) * self.output_scale


class SkipDecoderHead(nn.Module):
    def __init__(self, n_basis, hidden_dim, n_outputs, dropout, name=''):
        super().__init__()
        self.name = name
        self.n_outputs = n_outputs
        self.net = nn.Sequential(
            nn.Linear(n_basis, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_outputs))
        self.bias = nn.Parameter(torch.zeros(n_outputs))
        self.output_scale = nn.Parameter(torch.ones(n_outputs) * 0.01)

    def forward(self, z):
        return self.bias + self.net(z) * self.output_scale


class FederatedDeepMMNet(nn.Module):
    HEAD_NAMES = ['G_T', 'G_V', 'G_p', 'G_Vs', 'G_ps', 'G_W']

    def __init__(self, n_inputs, n_dynamic, column_info, config_dict):
        super().__init__()
        self.n_dynamic = n_dynamic
        n_basis = config_dict.get('N_BASIS', 128)
        branch_h = config_dict.get('BRANCH_HIDDEN', 128)
        tbranch_h = config_dict.get('TBRANCH_HIDDEN', 128)
        trunk_h = config_dict.get('TRUNK_HIDDEN', 64)
        n_fourier = config_dict.get('N_FOURIER_FREQ', 8)
        lstm_layers = config_dict.get('LSTM_LAYERS', 2)
        dropout = config_dict.get('DROPOUT', 0.3)
        dec_h = config_dict.get('DECODER_HIDDEN', 256)
        dec_h_s = config_dict.get('DECODER_HIDDEN_SMALL', 128)
        pred_steps = config_dict.get('PREDICTION_STEPS', 1)

        n_T = len(column_info['temp_indices'])
        n_V = len(column_info['flow_indices'])
        n_p = len(column_info['pressure_indices'])
        n_Vs = len(column_info['flow_sec_indices'])
        n_ps = len(column_info['pressure_sec_indices'])
        n_W = len(column_info['power_indices'])

        self.temp_indices = column_info['temp_indices']
        self.flow_indices = column_info['flow_indices']
        self.pressure_indices = column_info['pressure_indices']
        self.flow_sec_indices = column_info['flow_sec_indices']
        self.pressure_sec_indices = column_info['pressure_sec_indices']
        self.power_indices = column_info['power_indices']

        self.branch = BranchNetwork(n_inputs, branch_h, lstm_layers, n_basis, dropout)
        self.tbranch = TBranchNetwork(n_dynamic, tbranch_h, lstm_layers, n_basis, dropout)
        self.trunk = FourierTrunkNetwork(n_fourier, trunk_h, n_basis, pred_steps)

        self.head_T = DecoderHead(n_basis, dec_h, n_T, dropout, 'G_T')
        self.head_V = DecoderHead(n_basis, dec_h, n_V, dropout, 'G_V')
        self.head_p = DecoderHead(n_basis, dec_h, n_p, dropout, 'G_p')
        self.head_Vs = SkipDecoderHead(n_basis, dec_h_s, n_Vs, dropout, 'G_Vs')
        self.head_ps = SkipDecoderHead(n_basis, dec_h_s, n_ps, dropout, 'G_ps')
        self.head_W = SkipDecoderHead(n_basis, dec_h_s, n_W, dropout, 'G_W')

        self._heads = {'G_T': self.head_T, 'G_V': self.head_V, 'G_p': self.head_p,
                       'G_Vs': self.head_Vs, 'G_ps': self.head_ps, 'G_W': self.head_W}
        self._index_map = {'G_T': self.temp_indices, 'G_V': self.flow_indices,
                           'G_p': self.pressure_indices, 'G_Vs': self.flow_sec_indices,
                           'G_ps': self.pressure_sec_indices, 'G_W': self.power_indices}

    def forward(self, u_hist, y_hist, K=None):
        b = self.branch(u_hist)
        tb = self.tbranch(y_hist)
        T = self.trunk(K=K)
        z = b.unsqueeze(1) * tb.unsqueeze(1) * T.unsqueeze(0)
        B, K_actual, _ = z.shape
        predictions = torch.zeros(B, K_actual, self.n_dynamic, device=z.device)
        for hname in self.HEAD_NAMES:
            pred = self._heads[hname](z)
            indices = self._index_map[hname]
            predictions[:, :, indices] = pred
        return predictions


# ─── Phase 4: Domain DeepONet ─────────────────────────────────────────────────

class DomainDeepONet(nn.Module):
    def __init__(self, input_size, output_size, prediction_steps, config_dict):
        super().__init__()
        self.output_size = output_size
        self.prediction_steps = prediction_steps
        branch_h = config_dict.get('BRANCH_HIDDEN', 128)
        trunk_h = config_dict.get('TRUNK_HIDDEN', 64)
        n_basis = config_dict.get('N_BASIS', 32)
        n_fourier = config_dict.get('N_FOURIER_FREQ', 8)
        lstm_layers = config_dict.get('LSTM_LAYERS', 2)
        dropout = config_dict.get('DROPOUT', 0.3)
        use_skip = config_dict.get('USE_SKIP_CONNECTION', False)
        skip_alpha = config_dict.get('INITIAL_SKIP_ALPHA', 0.5)

        self.branch_encoder = nn.LSTM(input_size=input_size, hidden_size=branch_h,
                                       num_layers=lstm_layers, batch_first=True,
                                       dropout=dropout if lstm_layers > 1 else 0)
        self.branch_norm = nn.LayerNorm(branch_h)
        self.attention = nn.Sequential(
            nn.Linear(branch_h, branch_h // 2), nn.Tanh(),
            nn.Linear(branch_h // 2, 1))
        self.branch_head = nn.Sequential(
            nn.Linear(branch_h, branch_h), nn.LayerNorm(branch_h),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(branch_h, n_basis * output_size))

        self.register_buffer('query_times', torch.linspace(0, 1, prediction_steps).view(-1, 1))
        self.register_buffer('freqs', torch.linspace(1, n_fourier, n_fourier) * np.pi)
        trunk_in = 1 + 2 * n_fourier
        self.trunk_net = nn.Sequential(
            nn.Linear(trunk_in, trunk_h), nn.GELU(),
            nn.Linear(trunk_h, trunk_h), nn.GELU(),
            nn.Linear(trunk_h, n_basis))

        self.n_basis = n_basis
        if use_skip:
            init_logit = np.log(skip_alpha / (1 - skip_alpha + 1e-8))
            self.skip_alpha_logit = nn.Parameter(torch.ones(output_size) * init_logit)
        else:
            self.skip_alpha_logit = None
        self.output_scale = nn.Parameter(torch.ones(output_size) * 0.1)
        self.output_bias = nn.Parameter(torch.zeros(output_size))

    def forward(self, x):
        B = x.shape[0]
        lstm_out, _ = self.branch_encoder(x)
        lstm_out = self.branch_norm(lstm_out)
        w = torch.softmax(self.attention(lstm_out), dim=1)
        ctx = torch.sum(lstm_out * w, dim=1)
        branch_out = self.branch_head(ctx).view(B, self.n_basis, self.output_size)
        t = self.query_times
        feats = torch.cat([t, torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        trunk_out = self.trunk_net(feats)
        out = torch.einsum('pn,bno->bpo', trunk_out, branch_out)
        out = out * self.output_scale + self.output_bias
        if self.skip_alpha_logit is not None:
            alpha = torch.sigmoid(self.skip_alpha_logit)
            out = alpha.view(1, 1, -1) * out
        return {'predictions': out}


print("Model architectures defined for all 5 phases.")

Model architectures defined for all 5 phases.


In [4]:
#  Load Saved Models

def load_phase1_model(path, device):
    """Load Phase 1 Baseline LSTM."""
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    model = BaselineLSTM(
        input_size=cfg['total_input_size'],
        output_size=cfg['n_outputs'],
        hidden_size=cfg.get('HIDDEN_SIZE', 128),
        num_layers=cfg.get('NUM_LAYERS', 2),
        dropout=cfg.get('DROPOUT', 0.2),
        prediction_steps=cfg.get('PREDICTION_STEPS', 1),
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, cfg


def load_phase2_model(path, device):
    """Load Phase 2 Basic DeepONet."""
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    model = HybridDeepONet_P2(
        temporal_input_size=cfg['temporal_input_size'],
        temporal_output_size=cfg['temporal_output_size'],
        algebraic_input_size=cfg['algebraic_input_size'],
        algebraic_output_size=cfg['algebraic_output_size'],
        prediction_steps=cfg.get('PREDICTION_STEPS', 1),
        config_dict=cfg,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, cfg


def load_phase4_models(model_dir, device):
    """Load Phase 4 domain-specific models."""
    models = {}
    configs = {}
    domain_names = ['temperature', 'flow', 'pressure', 'power']
    for domain in domain_names:
        path = os.path.join(model_dir, f'{domain}_model.pt')
        if not os.path.exists(path):
            print(f"  Warning: {path} not found, skipping {domain}")
            continue
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        cfg = ckpt['config']
        n_in = ckpt.get('n_inputs', 515)
        n_out = ckpt.get('n_outputs', 257)
        # Infer input size from first layer
        input_size = n_in + n_out  # input + output history concatenated
        
        model = DomainDeepONet(
            input_size=input_size,
            output_size=n_out,
            prediction_steps=cfg.get('PREDICTION_STEPS', 4),
            config_dict=cfg,
        ).to(device)
        try:
            model.load_state_dict(ckpt['model_state_dict'])
        except RuntimeError:
            # Shape mismatch — try to infer input_size from state dict
            first_key = [k for k in ckpt['model_state_dict'] if 'branch_encoder.weight_ih_l0' in k]
            if first_key:
                input_size = ckpt['model_state_dict'][first_key[0]].shape[1]
                model = DomainDeepONet(
                    input_size=input_size, output_size=n_out,
                    prediction_steps=cfg.get('PREDICTION_STEPS', 4),
                    config_dict=cfg).to(device)
                model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        models[domain] = model
        configs[domain] = cfg
    return models, configs


def load_phase5_model(path, device):
    """Load Phase 5 Federated DeepM&Mnet."""
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    col_info = ckpt['column_info']
    n_inputs = ckpt['n_inputs']
    n_dynamic = ckpt['n_dynamic']
    
    model = FederatedDeepMMNet(
        n_inputs=n_inputs,
        n_dynamic=n_dynamic,
        column_info=col_info,
        config_dict=cfg,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, cfg, col_info


# Load all available models
loaded_models = {}

print("Loading saved models...")
for phase, path, loader in [
    ('Phase 1: LSTM', bench_config.P1_MODEL_PATH, lambda p: load_phase1_model(p, DEVICE)),
    ('Phase 2: DeepONet', bench_config.P2_MODEL_PATH, lambda p: load_phase2_model(p, DEVICE)),
]:
    if os.path.exists(path):
        try:
            result = loader(path)
            loaded_models[phase] = result
            n_params = sum(p.numel() for p in result[0].parameters())
            print(f"  ✓ {phase}: {n_params:,} params")
        except Exception as e:
            print(f"  ✗ {phase}: {e}")
    else:
        print(f"  ⊘ {phase}: file not found ({path})")

# Phase 3
if os.path.exists(bench_config.P3_MODEL_PATH):
    try:
        # Phase 3 has same structure as Phase 2 but with extra params
        ckpt3 = torch.load(bench_config.P3_MODEL_PATH, map_location='cpu', weights_only=False)
        cfg3 = ckpt3['config']
        # Use Phase 2 loader architecture (simplified — just time the forward pass)
        loaded_models['Phase 3: DeepONet+Fixes'] = (None, cfg3)  # placeholder
        print(f"  ✓ Phase 3: checkpoint loaded (config only for timing)")
    except Exception as e:
        print(f"  ✗ Phase 3: {e}")

# Phase 4
if os.path.isdir(bench_config.P4_MODEL_DIR):
    try:
        p4_models, p4_configs = load_phase4_models(bench_config.P4_MODEL_DIR, DEVICE)
        if p4_models:
            loaded_models['Phase 4: Domain-Specific'] = (p4_models, p4_configs)
            total_p4 = sum(sum(p.numel() for p in m.parameters()) for m in p4_models.values())
            print(f"  ✓ Phase 4: {len(p4_models)} domain models, {total_p4:,} total params")
    except Exception as e:
        print(f"  ✗ Phase 4: {e}")

# Phase 5
if os.path.exists(bench_config.P5_MODEL_PATH):
    try:
        p5_model, p5_cfg, p5_col_info = load_phase5_model(bench_config.P5_MODEL_PATH, DEVICE)
        loaded_models['Phase 5: Federated'] = (p5_model, p5_cfg, p5_col_info)
        n_params_5 = sum(p.numel() for p in p5_model.parameters())
        print(f"  ✓ Phase 5: {n_params_5:,} params")
    except Exception as e:
        print(f"  ✗ Phase 5: {e}")

print(f"\nLoaded {len(loaded_models)} models.")

Loading saved models...
  ✗ Phase 1: LSTM: cuDNN version incompatibility: PyTorch was compiled  against (9, 8, 0) but found runtime version (9, 4, 0). PyTorch already comes bundled with cuDNN. One option to resolving this error is to ensure PyTorch can find the bundled cuDNN. Looks like your LD_LIBRARY_PATH contains incompatible version of cudnn. Please either remove it from the path or install cudnn (9, 8, 0)
  ✓ Phase 2: DeepONet: 5,597,397 params
  ✓ Phase 3: checkpoint loaded (config only for timing)
  ✓ Phase 4: 4 domain models, 15,236,607 total params
  ✓ Phase 5: 3,113,756 params

Loaded 4 models.


In [11]:
#  FMU Simulator Timing

def time_fmu_simulator(n_steps_list, step_size=1, n_repeats=3):
    """
    Measure FMU simulator wall-clock time.
    
    Since the FMU is sequential (1 step at a time), we measure the time
    to simulate different numbers of prediction steps equivalent to 
    what the surrogate models predict in one forward pass.
    
    For K=1 at 30s subsample: 1 surrogate step = 30 FMU steps.
    """
    fmu_times = {}
    
    try:
        from fmu2ml.simulation.fmu_simulator import FMUSimulator
        
        print("Timing FMU simulator...")
        sim = FMUSimulator(system_name="summit")
        
        # Load a sample input row
        df = pd.read_parquet(bench_config.DATA_PATH)
        sample_row = df.iloc[0].to_dict()
        
        for n_steps in n_steps_list:
            times = []
            for _ in range(n_repeats):
                # Reset FMU for clean timing
                sim.reset()
                
                start = time.perf_counter()
                current_time = 0  # Track simulation time
                
                for step_idx in range(n_steps):
                    fmu_inputs = sim.model.generate_fmu_inputs(sample_row, uncertainties=False)
                    _, fmu_output = sim.model.step(current_time, fmu_inputs, step_size)
                    current_time += step_size  # Increment time
                    
                elapsed = time.perf_counter() - start
                times.append(elapsed)
            
            fmu_times[n_steps] = {
                'mean': np.mean(times),
                'std': np.std(times),
                'min': np.min(times),
                'per_step': np.mean(times) / n_steps,
            }
            print(f"  FMU {n_steps} steps: {np.mean(times):.4f}s "
                  f"({np.mean(times)/n_steps*1000:.2f} ms/step)")
        
        sim.cleanup()
        return fmu_times
        
    except ImportError:
        print("  FMU simulator not available — using estimated timing.")
        print("  (Based on typical FMU step times of ~5-15 ms/step)")
        # Use conservative estimate: ~8 ms per step for Modelica FMU
        FMU_MS_PER_STEP = 8.0
        for n_steps in n_steps_list:
            est_time = n_steps * FMU_MS_PER_STEP / 1000.0
            fmu_times[n_steps] = {
                'mean': est_time,
                'std': est_time * 0.1,
                'min': est_time * 0.9,
                'per_step': FMU_MS_PER_STEP / 1000.0,
            }
        print(f"  Estimated FMU timing: {FMU_MS_PER_STEP:.1f} ms/step")
        return fmu_times
    
    except Exception as e:
        print(f"  FMU timing failed: {e}")
        print("  Using estimated timing (8 ms/step)")
        FMU_MS_PER_STEP = 8.0
        for n_steps in n_steps_list:
            est_time = n_steps * FMU_MS_PER_STEP / 1000.0
            fmu_times[n_steps] = {
                'mean': est_time,
                'std': est_time * 0.1,
                'min': est_time * 0.9,
                'per_step': FMU_MS_PER_STEP / 1000.0,
            }
        return fmu_times


# The FMU equivalent for one surrogate prediction step:
# K=1 prediction at 30s subsample = 30 actual FMU steps
# For K=4 (Phase 4) at 30s subsample = 120 FMU steps
fmu_steps_per_surrogate = {
    'Phase 1: LSTM': 30,       # K=1, dt=30s
    'Phase 2: DeepONet': 30,   # K=1, dt=30s
    'Phase 3: DeepONet+Fixes': 30,
    'Phase 4: Domain-Specific': 120,  # K=4, dt=30s
    'Phase 5: Federated': 30,  # K=1, dt=30s
}

# Time FMU for the relevant step counts
unique_fmu_steps = sorted(set(fmu_steps_per_surrogate.values()))
fmu_timing = time_fmu_simulator(unique_fmu_steps, n_repeats=5)



Timing FMU simulator...
Initializing FMU...


2026-03-24 19:28:49 - fmu2ml.simulation.fmu_simulator - INFO - ✓ FMU simulator initialized for summit
2026-03-24 19:28:49 - fmu2ml.simulation.fmu_simulator - INFO - NUM_CDUS: 257


RK-method: sdirk34hw
Local extrapolation
FSAL
Continuous extension
Initializing FMU...


2026-03-24 19:29:26 - fmu2ml.simulation.fmu_simulator - INFO - FMU reset to initial state


RK-method: sdirk34hw
Local extrapolation
FSAL
Continuous extension
Initializing FMU...


...

RK-method: sdirk34hw
Local extrapolation
FSAL
Continuous extension
Initializing FMU...


2026-03-24 19:46:44 - fmu2ml.simulation.fmu_simulator - INFO - FMU reset to initial state


RK-method: sdirk34hw
Local extrapolation
FSAL
Continuous extension


2026-03-24 19:48:29 - fmu2ml.simulation.fmu_simulator - INFO - FMU resources cleaned up


  FMU 120 steps: 104.9074s (874.23 ms/step)


In [12]:
#  Surrogate Model Inference Timing

def generate_dummy_input(batch_size, seq_len, feature_dim, device):
    """Generate random input tensor for benchmarking."""
    return torch.randn(batch_size, seq_len, feature_dim, device=device)


def time_model_inference(model, input_fn, batch_sizes, n_warmup=5, n_repeats=20, device=DEVICE):
    """
    Time model inference across batch sizes.
    
    Args:
        model: nn.Module
        input_fn: function(batch_size) -> tuple of input tensors or single tensor
        batch_sizes: list of batch sizes to test
        n_warmup: warmup iterations
        n_repeats: timing iterations
    
    Returns:
        dict: {batch_size: {mean, std, min, per_sample, throughput}}
    """
    model.eval()
    results = {}
    
    for bs in batch_sizes:
        inputs = input_fn(bs)
        
        # Warmup (compile kernels, fill caches)
        with torch.no_grad():
            for _ in range(n_warmup):
                if isinstance(inputs, tuple):
                    _ = model(*inputs)
                else:
                    _ = model(inputs)
        
        # Synchronize GPU before timing
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        times = []
        for _ in range(n_repeats):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            start = time.perf_counter()
            with torch.no_grad():
                if isinstance(inputs, tuple):
                    _ = model(*inputs)
                else:
                    _ = model(inputs)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            elapsed = time.perf_counter() - start
            times.append(elapsed)
        
        times = np.array(times)
        results[bs] = {
            'mean': float(np.mean(times)),
            'std': float(np.std(times)),
            'min': float(np.min(times)),
            'per_sample': float(np.mean(times) / bs),
            'throughput': float(bs / np.mean(times)),  # samples/sec
        }
    
    return results


# Time each loaded model
surrogate_timing = {}
batch_sizes = bench_config.BATCH_SIZES

print("\nTiming surrogate model inference...")
print(f"Batch sizes: {batch_sizes}")
print(f"Warmup: {bench_config.N_WARMUP}, Repeats: {bench_config.N_REPEATS}")
print()

# Phase 1
if 'Phase 1: LSTM' in loaded_models:
    model_p1, cfg_p1 = loaded_models['Phase 1: LSTM']
    input_size_p1 = cfg_p1['total_input_size']
    hist_steps_p1 = cfg_p1.get('HISTORY_STEPS', 40)
    
    def p1_input_fn(bs):
        return generate_dummy_input(bs, hist_steps_p1, input_size_p1, DEVICE)
    
    surrogate_timing['Phase 1: LSTM'] = time_model_inference(
        model_p1, p1_input_fn, batch_sizes,
        bench_config.N_WARMUP, bench_config.N_REPEATS)
    print(f"Phase 1 (LSTM):")
    for bs, t in surrogate_timing['Phase 1: LSTM'].items():
        print(f"  BS={bs:5d}: {t['mean']*1000:8.2f} ms  ({t['throughput']:,.0f} samples/s)")

# Phase 2
if 'Phase 2: DeepONet' in loaded_models:
    model_p2, cfg_p2 = loaded_models['Phase 2: DeepONet']
    t_in = cfg_p2['temporal_input_size']
    a_in = cfg_p2['algebraic_input_size']
    hist_p2 = cfg_p2.get('HISTORY_STEPS', 40)
    pred_p2 = cfg_p2.get('PREDICTION_STEPS', 1)
    
    def p2_input_fn(bs):
        x_temp = generate_dummy_input(bs, hist_p2, t_in, DEVICE)
        x_alg = generate_dummy_input(bs, pred_p2, a_in, DEVICE)
        return (x_temp, x_alg)
    
    surrogate_timing['Phase 2: DeepONet'] = time_model_inference(
        model_p2, p2_input_fn, batch_sizes,
        bench_config.N_WARMUP, bench_config.N_REPEATS)
    print(f"\nPhase 2 (DeepONet):")
    for bs, t in surrogate_timing['Phase 2: DeepONet'].items():
        print(f"  BS={bs:5d}: {t['mean']*1000:8.2f} ms  ({t['throughput']:,.0f} samples/s)")

# Phase 4
if 'Phase 4: Domain-Specific' in loaded_models:
    p4_models, p4_configs = loaded_models['Phase 4: Domain-Specific']
    
    # Time the most representative domain (temperature — largest)
    for domain_name, domain_model in p4_models.items():
        # Infer input size from first LSTM layer
        first_param = next(domain_model.branch_encoder.parameters())
        d_in = first_param.shape[1]
        d_pred = domain_model.prediction_steps
        
        # Use history of 40 steps (typical)
        def p4_input_fn(bs, dim=d_in):
            return generate_dummy_input(bs, 40, dim, DEVICE)
        
        key = f'Phase 4: {domain_name.capitalize()}'
        surrogate_timing[key] = time_model_inference(
            domain_model, p4_input_fn, batch_sizes,
            bench_config.N_WARMUP, bench_config.N_REPEATS)
        print(f"\n{key}:")
        for bs, t in surrogate_timing[key].items():
            print(f"  BS={bs:5d}: {t['mean']*1000:8.2f} ms  ({t['throughput']:,.0f} samples/s)")

    # Combined Phase 4 timing (all 4 domains sequentially)
    def p4_combined_fn(bs):
        inputs = {}
        for dn, dm in p4_models.items():
            fp = next(dm.branch_encoder.parameters())
            inputs[dn] = generate_dummy_input(bs, 40, fp.shape[1], DEVICE)
        return inputs
    
    # Manual timing for combined
    p4_combined_times = {}
    for bs in batch_sizes:
        inputs = p4_combined_fn(bs)
        # Warmup
        for _ in range(bench_config.N_WARMUP):
            with torch.no_grad():
                for dn, dm in p4_models.items():
                    _ = dm(inputs[dn])
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        
        times = []
        for _ in range(bench_config.N_REPEATS):
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            with torch.no_grad():
                for dn, dm in p4_models.items():
                    _ = dm(inputs[dn])
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            times.append(time.perf_counter() - start)
        
        times = np.array(times)
        p4_combined_times[bs] = {
            'mean': float(np.mean(times)),
            'std': float(np.std(times)),
            'min': float(np.min(times)),
            'per_sample': float(np.mean(times) / bs),
            'throughput': float(bs / np.mean(times)),
        }
    surrogate_timing['Phase 4: Domain-Specific'] = p4_combined_times
    print(f"\nPhase 4 (Combined):")
    for bs, t in p4_combined_times.items():
        print(f"  BS={bs:5d}: {t['mean']*1000:8.2f} ms  ({t['throughput']:,.0f} samples/s)")

# Phase 5
if 'Phase 5: Federated' in loaded_models:
    p5_model, p5_cfg, p5_col_info = loaded_models['Phase 5: Federated']
    n_in_5 = p5_cfg.get('n_inputs', 515)  # fallback
    # Try to get from checkpoint
    n_in_5 = loaded_models['Phase 5: Federated'][1].get('n_inputs',
              sum(1 for _ in p5_model.branch.lstm.parameters()) and 515)
    n_dyn_5 = p5_model.n_dynamic
    hist_p5 = 40
    
    # Get actual input size from branch LSTM
    branch_lstm_param = next(p5_model.branch.lstm.parameters())
    n_in_5 = branch_lstm_param.shape[1]
    
    def p5_input_fn(bs):
        u = generate_dummy_input(bs, hist_p5, n_in_5, DEVICE)
        y = generate_dummy_input(bs, hist_p5, n_dyn_5, DEVICE)
        return (u, y)
    
    surrogate_timing['Phase 5: Federated'] = time_model_inference(
        p5_model, p5_input_fn, batch_sizes,
        bench_config.N_WARMUP, bench_config.N_REPEATS)
    print(f"\nPhase 5 (Federated):")
    for bs, t in surrogate_timing['Phase 5: Federated'].items():
        print(f"  BS={bs:5d}: {t['mean']*1000:8.2f} ms  ({t['throughput']:,.0f} samples/s)")

print("\n✓ All surrogate models timed.")


Timing surrogate model inference...
Batch sizes: [1, 4, 16, 64, 256, 1024, 2048, 4096]
Warmup: 5, Repeats: 20


Phase 2 (DeepONet):
  BS=    1:     1.20 ms  (832 samples/s)
  BS=    4:     1.26 ms  (3,177 samples/s)
  BS=   16:     1.25 ms  (12,818 samples/s)
  BS=   64:     1.27 ms  (50,437 samples/s)
  BS=  256:     2.72 ms  (94,151 samples/s)
  BS= 1024:     8.82 ms  (116,135 samples/s)
  BS= 2048:    17.17 ms  (119,293 samples/s)
  BS= 4096:    33.97 ms  (120,576 samples/s)

Phase 4: Temperature:
  BS=    1:     0.96 ms  (1,046 samples/s)
  BS=    4:     1.02 ms  (3,939 samples/s)
  BS=   16:     1.03 ms  (15,593 samples/s)
  BS=   64:     1.39 ms  (45,941 samples/s)
  BS=  256:     4.18 ms  (61,291 samples/s)
  BS= 1024:    14.58 ms  (70,234 samples/s)
  BS= 2048:    28.49 ms  (71,883 samples/s)
  BS= 4096:    56.56 ms  (72,419 samples/s)

Phase 4: Flow:
  BS=    1:     0.93 ms  (1,070 samples/s)
  BS=    4:     1.00 ms  (4,003 samples/s)
  BS=   16:     1.01 ms  (15,905 samples/

In [13]:
#  Compute Speedups

def compute_speedups(fmu_timing, surrogate_timing, fmu_steps_map, num_cdus=257):
    """
    Compute speedup = FMU_time / Surrogate_time.
    
    The FMU processes one CDU at a time internally, but outputs all CDUs per step.
    The surrogate also predicts all CDUs in one forward pass.
    
    Fair comparison: Time to produce the same prediction
    (one surrogate forward pass = N FMU steps for the same time horizon)
    """
    speedup_results = {}
    
    for phase_name, phase_timing in surrogate_timing.items():
        # Find how many FMU steps this phase replaces
        base_phase = phase_name.split(':')[0].strip() + ': ' + phase_name.split(': ')[1].split()[0]
        
        # Match to fmu_steps_per_surrogate
        fmu_steps = None
        for key, steps in fmu_steps_map.items():
            if key in phase_name or phase_name in key:
                fmu_steps = steps
                break
        
        if fmu_steps is None:
            # Default to 30 steps (K=1, dt=30s)
            fmu_steps = 30
        
        fmu_time = fmu_timing.get(fmu_steps, {}).get('mean', fmu_steps * 0.008)
        
        speedups = {}
        for bs, s_time in phase_timing.items():
            surrogate_time = s_time['mean']
            # Speedup: how many times faster is surrogate vs FMU for the same work
            # FMU time for one prediction = fmu_time (all CDUs in one go)
            # Surrogate time for batch of bs predictions = surrogate_time
            # Speedup per sample: fmu_time / (surrogate_time / bs) = fmu_time * bs / surrogate_time
            # But for fair comparison, 1 sample = 1 full prediction (all CDUs)
            speedup = fmu_time / (surrogate_time / bs) if surrogate_time > 0 else float('inf')
            
            # Another view: throughput speedup
            # FMU: 1/fmu_time predictions per second (sequential)
            # Surrogate: bs/surrogate_time predictions per second
            throughput_fmu = 1.0 / fmu_time
            throughput_surr = bs / surrogate_time
            throughput_speedup = throughput_surr / throughput_fmu
            
            speedups[bs] = {
                'surrogate_time_ms': surrogate_time * 1000,
                'fmu_time_ms': fmu_time * 1000,
                'fmu_steps': fmu_steps,
                'speedup_per_sample': fmu_time / s_time['per_sample'],
                'throughput_speedup': throughput_speedup,
                'surrogate_throughput': s_time['throughput'],
                'fmu_throughput': throughput_fmu,
            }
        
        speedup_results[phase_name] = speedups
    
    return speedup_results


speedup_results = compute_speedups(
    fmu_timing, surrogate_timing, fmu_steps_per_surrogate, bench_config.NUM_CDUS)

# Print speedup table
print("\n" + "=" * 90)
print("SPEEDUP RESULTS: Surrogate Models vs FMU Simulator")
print("=" * 90)

for phase_name, speedups in speedup_results.items():
    print(f"\n{phase_name}:")
    print(f"  {'Batch':>6s}  {'Surr (ms)':>10s}  {'FMU (ms)':>10s}  {'Speedup/sample':>15s}  {'Throughput Speedup':>18s}")
    print(f"  {'-'*6}  {'-'*10}  {'-'*10}  {'-'*15}  {'-'*18}")
    for bs, s in speedups.items():
        print(f"  {bs:6d}  {s['surrogate_time_ms']:10.2f}  {s['fmu_time_ms']:10.2f}  "
              f"{s['speedup_per_sample']:15.1f}×  {s['throughput_speedup']:18.1f}×")


SPEEDUP RESULTS: Surrogate Models vs FMU Simulator

Phase 2: DeepONet:
   Batch   Surr (ms)    FMU (ms)   Speedup/sample  Throughput Speedup
  ------  ----------  ----------  ---------------  ------------------
       1        1.20    65475.64          54462.1×             54462.1×
       4        1.26    65475.64         208014.5×            208014.5×
      16        1.25    65475.64         839234.1×            839234.1×
      64        1.27    65475.64        3302405.0×           3302405.0×
     256        2.72    65475.64        6164609.9×           6164609.9×
    1024        8.82    65475.64        7604035.2×           7604035.2×
    2048       17.17    65475.64        7810756.1×           7810756.1×
    4096       33.97    65475.64        7894780.4×           7894780.4×

Phase 4: Temperature:
   Batch   Surr (ms)    FMU (ms)   Speedup/sample  Throughput Speedup
  ------  ----------  ----------  ---------------  ------------------
       1        0.96    65475.64          68471.9

In [ ]:
#  Visualization 1 — Speedup vs Batch Size (All Phases)

os.makedirs('./surrogate_models/benchmark_vis', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

phase_colors = {
    'Phase 1: LSTM': '#1f77b4',
    'Phase 2: DeepONet': '#ff7f0e',
    'Phase 3: DeepONet+Fixes': '#2ca02c',
    'Phase 4: Domain-Specific': '#d62728',
    'Phase 5: Federated': '#9467bd',
}

# Left: Speedup per sample
ax = axes[0]
for phase_name, speedups in speedup_results.items():
    bs_list = sorted(speedups.keys())
    speedup_vals = [speedups[bs]['speedup_per_sample'] for bs in bs_list]
    color = phase_colors.get(phase_name, 'gray')
    # Shorten label
    label = phase_name.replace('Phase ', 'P')
    ax.plot(bs_list, speedup_vals, 'o-', label=label, color=color, 
            linewidth=2, markersize=8)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Speedup (×)', fontsize=12)
ax.set_title('Speedup per Sample vs FMU Simulator', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Break-even')
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())

# Right: Throughput speedup
ax = axes[1]
for phase_name, speedups in speedup_results.items():
    bs_list = sorted(speedups.keys())
    tp_vals = [speedups[bs]['throughput_speedup'] for bs in bs_list]
    color = phase_colors.get(phase_name, 'gray')
    label = phase_name.replace('Phase ', 'P')
    ax.plot(bs_list, tp_vals, 's-', label=label, color=color,
            linewidth=2, markersize=8)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Throughput Speedup (×)', fontsize=12)
ax.set_title('Throughput Speedup vs FMU Simulator', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())

plt.suptitle('Surrogate Model Speedup over FMU Simulator', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('./surrogate_models/benchmark_vis/speedup_vs_batchsize.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#  Visualization 3 — Per-Sample Latency Heatmap

fig, ax = plt.subplots(figsize=(14, 6))

# Build heatmap data: rows = phases, cols = batch sizes
phase_order = [p for p in [
    'Phase 1: LSTM', 'Phase 2: DeepONet', 'Phase 4: Domain-Specific', 'Phase 5: Federated'
] if p in surrogate_timing]

heatmap_data = []
row_labels = []
col_labels = [str(bs) for bs in bench_config.BATCH_SIZES]

for phase_name in phase_order:
    row = []
    for bs in bench_config.BATCH_SIZES:
        if bs in surrogate_timing[phase_name]:
            row.append(surrogate_timing[phase_name][bs]['per_sample'] * 1000)  # ms
        else:
            row.append(np.nan)
    heatmap_data.append(row)
    row_labels.append(phase_name.replace('Phase ', 'P'))

heatmap_arr = np.array(heatmap_data)

im = ax.imshow(heatmap_arr, aspect='auto', cmap='YlOrRd_r', 
               norm=plt.matplotlib.colors.LogNorm(
                   vmin=max(0.001, np.nanmin(heatmap_arr)),
                   vmax=np.nanmax(heatmap_arr)))

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=10)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Model Phase', fontsize=12)
ax.set_title('Per-Sample Latency (ms) — Lower is Better', fontsize=13)

# Annotate cells
for i in range(len(row_labels)):
    for j in range(len(col_labels)):
        val = heatmap_arr[i, j]
        if not np.isnan(val):
            color = 'white' if val > np.nanmedian(heatmap_arr) else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9, 
                    color=color, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Per-sample latency (ms)', fontsize=10)

plt.tight_layout()
plt.savefig('./surrogate_models/benchmark_vis/latency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#  Visualization 4 — Speedup Summary Bar Chart (at optimal batch size)

fig, ax = plt.subplots(figsize=(14, 7))

# Find best batch size for each phase (max throughput speedup)
phase_summaries = []

for phase_name, speedups in speedup_results.items():
    best_bs = max(speedups.keys(), key=lambda bs: speedups[bs]['throughput_speedup'])
    best = speedups[best_bs]
    phase_summaries.append({
        'Phase': phase_name.replace('Phase ', 'P'),
        'Best Batch Size': best_bs,
        'Speedup (per sample)': best['speedup_per_sample'],
        'Throughput Speedup': best['throughput_speedup'],
        'Surr Latency (ms)': best['surrogate_time_ms'] / best_bs * 1000,
        'FMU Latency (ms)': best['fmu_time_ms'],
    })

summary_df = pd.DataFrame(phase_summaries)

x = np.arange(len(summary_df))
width = 0.4

# Per-sample speedup
bars1 = ax.bar(x - width/2, summary_df['Speedup (per sample)'], width,
               label='Per-Sample Speedup', color='steelblue', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, summary_df['Throughput Speedup'], width,
               label='Throughput Speedup', color='coral', edgecolor='black', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(summary_df['Phase'], fontsize=10)
ax.set_ylabel('Speedup (×) over FMU Simulator', fontsize=12)
ax.set_title('Maximum Speedup at Optimal Batch Size', fontsize=14)
ax.legend(fontsize=11)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

# Annotate with actual values and batch sizes
for i, row in summary_df.iterrows():
    ax.annotate(f'{row["Throughput Speedup"]:.0f}×\n(BS={row["Best Batch Size"]})',
                xy=(i + width/2, row['Throughput Speedup']),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.annotate(f'{row["Speedup (per sample)"]:.0f}×',
                xy=(i - width/2, row['Speedup (per sample)']),
                ha='center', va='bottom', fontsize=9, fontweight='bold', color='darkblue')

plt.tight_layout()
plt.savefig('./surrogate_models/benchmark_vis/speedup_summary.png', dpi=150, bbox_inches='tight')
plt.show()